In [ ]:
!pip install pyautogen autogen google-generativeai --quiet

In [ ]:
import google.generativeai as genai
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os

# --- Configuration ---
# Ensure your Google API Key is set as an environment variable
# For security, avoid hardcoding API keys directly in the script.
# If running in an environment where __initial_auth_token is available,
# it might be used for authentication. For this specific Autogen setup,
# we rely on GOOGLE_API_KEY.
# os.environ["GOOGLE_API_KEY"] = "Your_key" # User needs to set this externally or replace "Your_key"
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

# LLM configuration for Autogen agents
LLM_CONFIG = {
    "config_list": [
        {
            "model": "gemini-1.5-flash",
            "api_key": os.environ.get("GOOGLE_API_KEY"),
            "api_type": "google"
        }
    ]
}

# --- State Management Class ---
class StateFlow:
    """
    Manages the investment strategy determination based on portfolio data.
    """
    def __init__(self):
        self.investment_type = None

    def determine_strategy(self, portfolio_data: dict) -> str:
        """
        Determines if the user should pursue a "GROWTH" or "VALUE" investment strategy
        based on their total investment relative to their salary.

        Args:
            portfolio_data (dict): A dictionary containing 'fd', 'sips', 'real_estate', and 'salary'.

        Returns:
            str: "GROWTH" if total investment is less than twice the salary, "VALUE" otherwise.
        """
        # Calculate total investment from provided portfolio data
        total_investment = portfolio_data.get('fd', 0) + \
                           portfolio_data.get('sips', 0) + \
                           portfolio_data.get('real_estate', 0)
        salary = portfolio_data.get('salary', 0)

        # Determine strategy based on a simple heuristic
        if total_investment < salary * 2:
            self.investment_type = "GROWTH"
        else:
            self.investment_type = "VALUE"

        return self.investment_type

# --- Agent Definitions ---
# Portfolio Analysis Agent: Analyzes the portfolio and recommends a category.
portfolio_agent = AssistantAgent(
    name="Portfolio_Analysis_Agent",
    llm_config=LLM_CONFIG,
    system_message="""You are the Portfolio Analysis Agent.
    Analyze the user's investment portfolio and determine if they should pursue Growth or Value investments.

    After receiving portfolio data, provide:
    1. Portfolio summary
    2. Investment category recommendation (Growth or Value)
    3. Brief reasoning for the recommendation"""
)

# Growth Investment Agent: Provides high-growth investment recommendations.
growth_agent = AssistantAgent(
    name="Growth_Investment_Agent",
    llm_config=LLM_CONFIG,
    system_message="""You are the Growth Investment Agent.
    Provide high-growth investment recommendations:
    - Equity mutual funds
    - Growth stocks
    - High-risk high-reward options

    Give specific investment suggestions with expected returns."""
)

# Value Investment Agent: Provides stable, long-term investment recommendations.
value_agent = AssistantAgent(
    name="Value_Investment_Agent",
    llm_config=LLM_CONFIG,
    system_message="""You are the Value Investment Agent.
    Provide stable, long-term investment recommendations:
    - Debt funds
    - Fixed deposits
    - Dividend stocks
    - Conservative options

    Give specific investment suggestions focusing on stability."""
)

# Investment Advisor Agent: Compiles a comprehensive financial report.
advisor_agent = AssistantAgent(
    name="Investment_Advisor_Agent",
    llm_config=LLM_CONFIG,
    system_message="""You are the Investment Advisor Agent.
    Create a personalized financial report based on:
    - Portfolio analysis
    - Investment recommendations from Growth/Value agents

    Provide a comprehensive final report with actionable recommendations."""
)

# User Proxy Agent: Represents the user and initiates the process.
user_proxy = UserProxyAgent(
    name="User_Proxy_Agent",
    human_input_mode="NEVER",  # Set to "ALWAYS" for human interaction
    system_message="You represent the user and initiate the portfolio management process."
)

# --- Helper Functions ---
def collect_user_portfolio_data() -> dict:
    """
    Collects financial data from the user.

    Returns:
        dict: A dictionary containing salary, fixed deposits, SIPs, and real estate values.
    """
    print("=== Financial Portfolio Manager ===")
    portfolio_data = {}
    fields = {
        "Current salary": "salary",
        "Fixed Deposits amount": "fd",
        "SIPs amount": "sips",
        "Real Estate value": "real_estate"
    }

    for prompt, key in fields.items():
        while True:
            try:
                value = float(input(f"{prompt}: "))
                if value < 0:
                    print("Please enter a non-negative value.")
                    continue
                portfolio_data[key] = value
                break
            except ValueError:
                print("Invalid input. Please enter a numerical value.")
    return portfolio_data

def run_portfolio_manager():
    """
    Orchestrates the financial portfolio management process using Autogen agents.
    """
    # Step 1: Collect user inputs
    portfolio_data = collect_user_portfolio_data()

    # Step 2: StateFlow determines strategy
    state_flow = StateFlow()
    strategy = state_flow.determine_strategy(portfolio_data)
    print(f"\n✅ StateFlow Decision: {strategy} Investment Strategy")

    # Step 3: Select appropriate investment agent based on StateFlow's decision
    investment_agent = growth_agent if strategy == "GROWTH" else value_agent

    # Step 4: Create Group Chat with selected agents
    group_chat = GroupChat(
        agents=[user_proxy, portfolio_agent, investment_agent, advisor_agent],
        messages=[],
        max_round=8  # Maximum number of conversation rounds
    )

    group_chat_manager = GroupChatManager(
        groupchat=group_chat,
        llm_config=LLM_CONFIG
    )

    # Step 5: Prepare the initial message for the group chat
    total_portfolio_value = portfolio_data['fd'] + portfolio_data['sips'] + portfolio_data['real_estate']
    initial_message = f"""
    Portfolio Management Request:

    User Portfolio Data:
    - Salary: ₹{portfolio_data['salary']:,.2f}
    - Fixed Deposits: ₹{portfolio_data['fd']:,.2f}
    - SIPs: ₹{portfolio_data['sips']:,.2f}
    - Real Estate: ₹{portfolio_data['real_estate']:,.2f}
    - Total Portfolio Value: ₹{total_portfolio_value:,.2f}

    StateFlow has determined: {strategy} Investment Strategy

    Please proceed with the analysis and recommendations.
    """

    # Step 6: Execute Group Chat
    print("\n🚀 Starting Group Chat for Portfolio Management...")
    try:
        result = user_proxy.initiate_chat(
            recipient=group_chat_manager,
            message=initial_message,
            max_turns=10  # Increased max_turns for potentially longer conversations
        )
        print("\n✅ Portfolio Management Complete!")
        print(f"📊 Strategy Used: {strategy}")
        print(f"💼 Total Portfolio: ₹{total_portfolio_value:,.2f}")
    except Exception as e:
        print(f"\n❌ An error occurred during the group chat: {e}")
        print("Please ensure your GOOGLE_API_KEY is correctly set and try again.")


# --- Main Execution ---
if __name__ == "__main__":
    # Check if API key is set
    if not os.environ.get("GOOGLE_API_KEY"):
        print("Error: GOOGLE_API_KEY environment variable is not set.")
        print("Please set the GOOGLE_API_KEY environment variable before running the script.")
    else:
        run_portfolio_manager()
